In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import nest_asyncio

nest_asyncio.apply()  # used in notebook

from phyling.ble import NanoPhyling, MiniPhyling, PhylingDevices

In [ ]:
devices = PhylingDevices(
    [
        MiniPhyling(
            module_name="gauche",  # Optionnel, par défaut on utilise le nom Bluetooth
            ble_name="MiniPhyling_42"
        ),
        NanoPhyling(
            module_name="droite",  # Optionnel, par défaut on utilise le nom Bluetooth
            ble_name="NanoPhyling_42",
            config={"rate": 200, "data": ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]},
        ),
    ],
    output_fs=200,
)

## Calibration

`tare()` enregistre 5s de données immobiles par device (en parallèle) et calcule un offset par colonne.  
La calibration s'applique automatiquement dans `get_df()` : `coef * (val + offset)`.  
Le df existant de chaque device est **préservé**.

In [ ]:
# Tare sur plusieurs devices en parallèle
devices.tare({
    "droite": ["gyro_x", "gyro_y", "gyro_z"],
    "gauche": ["gyro_x", "gyro_y", "gyro_z"],
})

In [ ]:
# Voir la calibration de tous les devices
devices.get_calib()

In [ ]:
# Reset la calibration (vide le dict, pas de calibration)
devices.reset_calib()

In [ ]:
# Remplacer la calibration d'un device
devices.set_calib({
    "droite": {
        "mag_z": {"coef": 1, "offset": 0},
    },
})

In [ ]:
# Merger une calibration partielle
devices.update_calib({
    "droite": {
        "acc_z": {"coef": 1.02, "offset": 0},
    },
})

In [ ]:
# On peut aussi calibrer un device directement
droite = devices.get_device("droite")
droite.tare(["gyro_z"])
droite.get_calib()

## Acquisition

In [ ]:
# Vous pouvez kill la cell pendant l'exécution pour arrêter l'enregistrement
devices.run(duration=30)

In [ ]:
# La calibration est appliquée ici pour chaque device
df = devices.get_df(drop_nan=False)
df.head()

In [ ]:
cols = ["acc_x", "acc_y", "acc_z"]

fig, axes = plt.subplots(len(cols), 1, sharex=True, figsize=(8, 6))

for i, col in enumerate(cols):
    ax = axes[i]

    for device in devices.devices:
        colname = f"{device.get_name()}.{col}"
        if colname in df.columns:
            ax.plot(df["T"], df[colname], label=colname)

    ax.set_title(col)
    ax.grid()
    ax.legend()

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()

In [ ]:
# Enregistrer les données dans un CSV
# df.to_csv("example_data.csv", index=False)